# ScribblBot — Colab Training Notebook

Run every cell top to bottom. At the end your trained model files will be saved to Google Drive.

**Before running:** go to Runtime > Change runtime type > T4 GPU

In [ ]:
# 1. Install dependencies
!pip install scikit-image seaborn joblib --quiet

In [ ]:
# 2. Mount Google Drive so models persist after the session ends
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/scribblbot_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Models will be saved to: {SAVE_DIR}')

In [ ]:
# 3. Config
from pathlib import Path

CLASSES = [
    'cat', 'dog', 'pizza', 'bicycle', 'house',
    'sun', 'tree', 'car', 'fish', 'butterfly',
    'guitar', 'hamburger', 'airplane', 'banana', 'star',
]
NUM_CLASSES = len(CLASSES)

TRAIN_SAMPLES_PER_CLASS = 2000
TEST_SAMPLES_PER_CLASS = 400
IMG_SIZE = 28

DEEP_BATCH_SIZE = 256    # bigger batch since we have GPU
DEEP_EPOCHS = 15
DEEP_LR = 1e-3
DEEP_WEIGHT_DECAY = 1e-4

RF_N_ESTIMATORS = 200
HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (4, 4)
HOG_CELLS_PER_BLOCK = (2, 2)

EXPERIMENT_FRACTIONS = [0.1, 0.25, 0.5, 0.75, 1.0]
EXPERIMENT_EPOCHS = 10

RAW_DIR = Path('/content/data/raw')
PROCESSED_DIR = Path('/content/data/processed')
OUTPUTS_DIR = Path('/content/data/outputs')
MODELS_DIR = Path('/content/models')

for d in [RAW_DIR, PROCESSED_DIR, OUTPUTS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

QUICKDRAW_URL = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/{cls}.npy'

print(f'Config ready. {NUM_CLASSES} classes, {TRAIN_SAMPLES_PER_CLASS} train samples each.')

In [ ]:
# 4. Download Quick Draw data
import urllib.request

def download_class(cls):
    url = QUICKDRAW_URL.format(cls=cls.replace(' ', '%20'))
    dest = RAW_DIR / f'{cls}.npy'
    if dest.exists():
        print(f'  already have {cls}.npy')
        return
    urllib.request.urlretrieve(url, dest)
    print(f'  downloaded {cls}.npy')

print('Downloading dataset...')
for cls in CLASSES:
    download_class(cls)
print('Done.')

In [ ]:
# 5. Build train/test splits and HOG features
import numpy as np
from skimage.feature import hog

def load_class_data(cls, n_train, n_test):
    data = np.load(RAW_DIR / f'{cls}.npy', mmap_mode='r')
    rng = np.random.default_rng(seed=42)
    indices = rng.permutation(len(data))[:n_train + n_test]
    data = data[indices]
    return data[:n_train], data[n_train:n_train + n_test]

def extract_hog(pixel_matrix):
    features = []
    for row in pixel_matrix:
        img = row.reshape(IMG_SIZE, IMG_SIZE)
        desc = hog(img, orientations=HOG_ORIENTATIONS,
                   pixels_per_cell=HOG_PIXELS_PER_CELL,
                   cells_per_block=HOG_CELLS_PER_BLOCK,
                   visualize=False, channel_axis=None)
        features.append(desc)
    return np.array(features, dtype=np.float32)

print('Loading raw data...')
train_raws, test_raws, train_labels, test_labels = [], [], [], []
for idx, cls in enumerate(CLASSES):
    tr, te = load_class_data(cls, TRAIN_SAMPLES_PER_CLASS, TEST_SAMPLES_PER_CLASS)
    train_raws.append(tr)
    test_raws.append(te)
    train_labels.append(np.full(len(tr), idx, dtype=np.int64))
    test_labels.append(np.full(len(te), idx, dtype=np.int64))
    print(f'  {cls}')

X_train_raw = np.concatenate(train_raws)
X_test_raw = np.concatenate(test_raws)
y_train = np.concatenate(train_labels)
y_test = np.concatenate(test_labels)

rng = np.random.default_rng(seed=0)
perm = rng.permutation(len(X_train_raw))
X_train_raw = X_train_raw[perm]
y_train = y_train[perm]

print('\nExtracting HOG features (train)...')
X_train_hog = extract_hog(X_train_raw)
print('Extracting HOG features (test)...')
X_test_hog = extract_hog(X_test_raw)

np.save(PROCESSED_DIR / 'X_train_raw.npy', X_train_raw)
np.save(PROCESSED_DIR / 'X_test_raw.npy', X_test_raw)
np.save(PROCESSED_DIR / 'y_train.npy', y_train)
np.save(PROCESSED_DIR / 'y_test.npy', y_test)
np.save(PROCESSED_DIR / 'X_train_hog.npy', X_train_hog)
np.save(PROCESSED_DIR / 'X_test_hog.npy', X_test_hog)

print(f'\nTrain: {X_train_raw.shape}, Test: {X_test_raw.shape}, HOG features: {X_train_hog.shape[1]}')

In [ ]:
# 6. Naive baseline
from sklearn.metrics import accuracy_score
import joblib

majority_class = int(np.bincount(y_train).argmax())
naive_preds = np.full(len(y_test), majority_class, dtype=np.int64)
naive_acc = accuracy_score(y_test, naive_preds)

joblib.dump({'majority_class': majority_class, 'accuracy': naive_acc}, MODELS_DIR / 'naive_model.pkl')
print(f'Naive baseline accuracy: {naive_acc:.4f}')
print(f'Majority class: {CLASSES[majority_class]}')

In [ ]:
# 7. Classical ML: Random Forest on HOG features
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import time

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train_hog)
X_te_scaled = scaler.transform(X_test_hog)

rf = RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, n_jobs=-1, random_state=42)
t0 = time.time()
rf.fit(X_tr_scaled, y_train)
elapsed = time.time() - t0

rf_preds = rf.predict(X_te_scaled)
rf_acc = accuracy_score(y_test, rf_preds)

joblib.dump({'clf': rf, 'scaler': scaler}, MODELS_DIR / 'classical_model.pkl')

print(f'Random Forest trained in {elapsed:.1f}s')
print(f'Test accuracy: {rf_acc:.4f}')
print()
print(classification_report(y_test, rf_preds, target_names=CLASSES))

In [ ]:
# 8. Define ScribblNet
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

class ScribblNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

print('ScribblNet defined.')

In [ ]:
# 9. Train ScribblNet
def make_loaders(X_raw, y, X_test, y_test, batch_size, fraction=1.0):
    if fraction < 1.0:
        n = max(1, int(len(X_raw) * fraction))
        idx = np.random.default_rng(seed=7).permutation(len(X_raw))[:n]
        X_raw = X_raw[idx]
        y = y[idx]
    def to_ds(X, labels):
        imgs = torch.from_numpy(X.astype(np.float32) / 255.0).view(-1, 1, IMG_SIZE, IMG_SIZE)
        return TensorDataset(imgs, torch.from_numpy(labels))
    train_loader = DataLoader(to_ds(X_raw, y), batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader  = DataLoader(to_ds(X_test, y_test), batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, test_loader

def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            preds = model(imgs.to(device)).argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return accuracy_score(labels, preds), preds

train_loader, test_loader = make_loaders(X_train_raw, y_train, X_test_raw, y_test, DEEP_BATCH_SIZE)

model = ScribblNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=DEEP_LR, weight_decay=DEEP_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=DEEP_EPOCHS)
criterion = nn.CrossEntropyLoss()

best_acc = 0.0
history = {'loss': [], 'val_acc': []}

print(f'Training for {DEEP_EPOCHS} epochs...')
for epoch in range(1, DEEP_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    val_acc, _ = evaluate_model(model, test_loader)
    scheduler.step()
    history['loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    print(f'  epoch {epoch:02d}/{DEEP_EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.4f}')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODELS_DIR / 'deep_model.pth')

print(f'\nBest test accuracy: {best_acc:.4f}')

In [ ]:
# 10. Training curves
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(range(1, len(history['loss']) + 1), history['loss'],
         color='steelblue', marker='o', linestyle='solid', markersize=5)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('ScribblNet Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, len(history['val_acc']) + 1), history['val_acc'],
         color='seagreen', marker='o', linestyle='solid', markersize=5)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Accuracy')
ax2.set_title('ScribblNet Validation Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'deep_training_curves.png', dpi=150)
plt.show()

In [ ]:
# 11. Confusion matrices
import seaborn as sns

model.load_state_dict(torch.load(MODELS_DIR / 'deep_model.pth', map_location=device))
_, deep_preds = evaluate_model(model, test_loader)

def plot_cm(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='.2f', xticklabels=CLASSES,
                yticklabels=CLASSES, cmap='Blues', ax=ax, linewidths=0.5)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / filename, dpi=150)
    plt.show()

plot_cm(y_test, deep_preds, 'ScribblNet Confusion Matrix', 'deep_confusion_matrix.png')
plot_cm(y_test, rf_preds, 'Random Forest Confusion Matrix', 'classical_confusion_matrix.png')

In [ ]:
# 12. Model comparison bar chart
names = ['Naive Baseline', 'Random Forest', 'ScribblNet']
accs = [naive_acc, rf_acc, best_acc]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(names, accs, color=['#94a3b8', '#60a5fa', '#34d399'], width=0.5)
ax.set_ylim(0, 1)
ax.set_ylabel('Test Accuracy')
ax.set_title('Model Comparison')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{acc:.3f}', ha='center', fontsize=12)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'model_comparison.png', dpi=150)
plt.show()

In [ ]:
# 13. Experiment: training set size sensitivity
# Sweeps over fractions of training data for both models.
# Shows how data volume affects each approach.
print('Running experiment: training size sensitivity sweep...')

deep_accs_exp = []
rf_accs_exp = []
n_samples_list = []

for frac in EXPERIMENT_FRACTIONS:
    n = int(len(X_train_raw) * frac)
    n_samples_list.append(n)
    print(f'\nFraction {frac:.0%} (n={n})')

    # Deep model
    tr_loader, te_loader = make_loaders(X_train_raw, y_train, X_test_raw, y_test,
                                         DEEP_BATCH_SIZE, fraction=frac)
    exp_model = ScribblNet().to(device)
    exp_opt = torch.optim.Adam(exp_model.parameters(), lr=DEEP_LR, weight_decay=DEEP_WEIGHT_DECAY)
    exp_sched = torch.optim.lr_scheduler.CosineAnnealingLR(exp_opt, T_max=EXPERIMENT_EPOCHS)
    exp_model.train()
    for ep in range(EXPERIMENT_EPOCHS):
        for imgs, labels in tr_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            exp_opt.zero_grad()
            criterion(exp_model(imgs), labels).backward()
            exp_opt.step()
        exp_sched.step()
    acc_deep, _ = evaluate_model(exp_model, te_loader)
    deep_accs_exp.append(acc_deep)
    print(f'  ScribblNet acc: {acc_deep:.4f}')

    # Random Forest
    idx = np.random.default_rng(seed=42).permutation(len(X_train_hog))[:n]
    sc = StandardScaler()
    X_tr_exp = sc.fit_transform(X_train_hog[idx])
    X_te_exp = sc.transform(X_test_hog)
    rf_exp = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    rf_exp.fit(X_tr_exp, y_train[idx])
    acc_rf = accuracy_score(y_test, rf_exp.predict(X_te_exp))
    rf_accs_exp.append(acc_rf)
    print(f'  Random Forest acc: {acc_rf:.4f}')

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_samples_list, deep_accs_exp, marker='o', linestyle='solid',
        label='ScribblNet (CNN)', linewidth=2, markersize=7)
ax.plot(n_samples_list, rf_accs_exp, marker='s', linestyle='dashed',
        label='Random Forest (HOG)', linewidth=2, markersize=7)
ax.set_xlabel('Training samples')
ax.set_ylabel('Test accuracy')
ax.set_title('Training Set Size Sensitivity')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'experiment_sensitivity.png', dpi=150)
plt.show()

import json
with open(OUTPUTS_DIR / 'experiment_results.json', 'w') as f:
    json.dump({'fractions': EXPERIMENT_FRACTIONS, 'n_samples': n_samples_list,
               'deep_accs': deep_accs_exp, 'rf_accs': rf_accs_exp}, f, indent=2)

print('Experiment complete.')

In [ ]:
# 14. Save everything to Google Drive
import shutil

files_to_save = [
    (MODELS_DIR / 'deep_model.pth',       'deep_model.pth'),
    (MODELS_DIR / 'classical_model.pkl',  'classical_model.pkl'),
    (MODELS_DIR / 'naive_model.pkl',       'naive_model.pkl'),
    (OUTPUTS_DIR / 'deep_training_curves.png',   'deep_training_curves.png'),
    (OUTPUTS_DIR / 'deep_confusion_matrix.png',  'deep_confusion_matrix.png'),
    (OUTPUTS_DIR / 'classical_confusion_matrix.png', 'classical_confusion_matrix.png'),
    (OUTPUTS_DIR / 'model_comparison.png',       'model_comparison.png'),
    (OUTPUTS_DIR / 'experiment_sensitivity.png', 'experiment_sensitivity.png'),
    (OUTPUTS_DIR / 'experiment_results.json',    'experiment_results.json'),
]

for src, name in files_to_save:
    dest = Path(SAVE_DIR) / name
    shutil.copy(src, dest)
    print(f'Saved {name}')

print(f'\nAll files saved to Google Drive at: {SAVE_DIR}')
print('Download deep_model.pth and put it in your local models/ folder.')

In [ ]:
# 15. Results summary
print('Results Summary')
print(f'  Naive baseline:  {naive_acc:.4f}')
print(f'  Random Forest:   {rf_acc:.4f}')
print(f'  ScribblNet:      {best_acc:.4f}')

with open(OUTPUTS_DIR / 'results_summary.json', 'w') as f:
    json.dump({'naive_accuracy': naive_acc, 'classical_accuracy': rf_acc,
               'deep_accuracy': best_acc}, f, indent=2)
shutil.copy(OUTPUTS_DIR / 'results_summary.json', Path(SAVE_DIR) / 'results_summary.json')
print('results_summary.json saved to Drive.')